# consumer-agents — Run Analysis

Explore the events, snapshots, and behavior patterns from one simulation run.

**To use:** change `RUN_DIR` below to point at any run under `runs/`, then run all cells (Shift+Enter or Run All).

Sections:
1. Setup
2. Run summary (scenario + seed)
3. Event counts
4. Per-persona behavior
5. Purchases — what, where, how much
6. Top SKUs and behavioral streaks
7. Cash trajectory
8. Funnel — view → cart_add → purchase
9. Reflections (LLM self-talk)
10. Life events
11. Raw event browser


## 1. Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from consumer_agents.datalake.queries import open_run

# Point this at the run you want to analyze.
RUN_DIR = Path("../runs/baseline-d862c721")

con = open_run(RUN_DIR)

def q(sql: str) -> pd.DataFrame:
    """Run a SQL query against the run's tables and return a DataFrame."""
    return con.execute(sql).fetchdf()

print(f"Run:   {RUN_DIR}")
print(f"Files: {[p.name for p in RUN_DIR.iterdir() if p.is_file()]}")


## 2. Run summary

What scenario was this? `scenario.yaml` is an immutable copy of the YAML that produced this run.


In [ ]:
print((RUN_DIR / 'scenario.yaml').read_text())
print(f"\nSeed: {(RUN_DIR / 'seed.txt').read_text().strip()}")


## 3. Event counts

Tally of every event type in the run.

**LLM-generated event types:** `view`, `cart_add`, `purchase`, `abandon`, `return`, `reflection`.

**Scheduler-generated event types:** `income`, `expense`, `life_event` (scripted only in v0).


In [ ]:
q("""
  SELECT event_type, COUNT(*) AS n
  FROM events
  GROUP BY 1
  ORDER BY n DESC
""")


## 4. Per-persona behavior

Activity mix per persona.


In [ ]:
q("""
  SELECT
    agent_id,
    COUNT(*) FILTER (WHERE event_type = 'view')       AS views,
    COUNT(*) FILTER (WHERE event_type = 'cart_add')   AS cart_adds,
    COUNT(*) FILTER (WHERE event_type = 'purchase')   AS purchases,
    COUNT(*) FILTER (WHERE event_type = 'abandon')    AS abandons,
    COUNT(*) FILTER (WHERE event_type = 'reflection') AS reflections
  FROM events
  GROUP BY 1
  ORDER BY 1
""")


## 5. Purchases — what, where, how much

Each row in `v_purchases` is a checkout event emitted by the LLM via the `emit_actions` tool.


In [ ]:
q("""
  SELECT
    agent_id,
    COUNT(*)                 AS purchases,
    ROUND(SUM(total_usd), 2) AS spend_usd,
    ROUND(AVG(total_usd), 2) AS avg_basket_usd,
    ROUND(MAX(total_usd), 2) AS max_basket_usd
  FROM v_purchases
  GROUP BY 1
  ORDER BY 1
""")


### Spend split by retailer


In [ ]:
q("""
  SELECT
    agent_id,
    retailer,
    COUNT(*) AS purchases,
    ROUND(SUM(total_usd), 2) AS spend_usd
  FROM v_purchases
  GROUP BY 1, 2
  ORDER BY 1, spend_usd DESC
""")


### Category mix in cart_add events

What did each persona almost-buy by category? `cart_add` events let us see category engagement, which `v_purchases` flattens away (it just has cart totals).


In [ ]:
q("""
  SELECT
    agent_id,
    CASE
      WHEN sku_id LIKE 'sku-groc-%' THEN 'groceries'
      WHEN sku_id LIKE 'sku-elec-%' THEN 'electronics'
      WHEN sku_id LIKE 'sku-dine-%' THEN 'dining'
      ELSE 'other'
    END AS category,
    COUNT(*) AS cart_adds,
    ROUND(SUM(qty * unit_price_usd), 2) AS cart_value_usd
  FROM v_cart_add
  GROUP BY 1, 2
  ORDER BY 1, cart_value_usd DESC
""")


## 6. Top SKUs and behavioral streaks

The most-purchased SKU per persona, and how long the *longest unbroken daily-purchase streak* was. This is the section that surfaced the surprise in the 90-day baseline: all three personas bought the same coffee-shop drip coffee almost daily for ~80 days each.


### Top 5 SKUs purchased per persona


In [ ]:
q("""
  WITH ranked AS (
    SELECT
      agent_id,
      sku_id,
      SUM(qty) AS total_qty,
      COUNT(DISTINCT tick_day) AS days_bought,
      ROUND(SUM(qty * unit_price_usd), 2) AS total_spend,
      ROW_NUMBER() OVER (PARTITION BY agent_id ORDER BY COUNT(DISTINCT tick_day) DESC) AS rk
    FROM v_cart_add
    GROUP BY 1, 2
  )
  SELECT agent_id, sku_id, total_qty, days_bought, total_spend
  FROM ranked WHERE rk <= 5
  ORDER BY agent_id, days_bought DESC
""")


### Longest consecutive-day purchase streak per (agent, SKU)

Detects runs of consecutive days on which the same SKU was added to cart. The classic *islands-and-gaps* SQL pattern.


In [ ]:
q("""
  WITH days AS (
    SELECT DISTINCT agent_id, sku_id, tick_day
    FROM v_cart_add
  ),
  grouped AS (
    SELECT
      agent_id, sku_id, tick_day,
      tick_day - INTERVAL (ROW_NUMBER() OVER (PARTITION BY agent_id, sku_id ORDER BY tick_day)) DAY AS grp
    FROM days
  ),
  runs AS (
    SELECT
      agent_id, sku_id, grp,
      COUNT(*) AS streak_len,
      MIN(tick_day) AS streak_start,
      MAX(tick_day) AS streak_end
    FROM grouped
    GROUP BY 1, 2, 3
  )
  SELECT agent_id, sku_id, streak_len, streak_start, streak_end
  FROM runs
  WHERE streak_len >= 3
  ORDER BY streak_len DESC, agent_id
  LIMIT 20
""")


## 7. Cash trajectory

How does each persona's cash change over the run? From daily snapshots.


In [ ]:
cash = q("""
  SELECT agent_id, tick_day, cash_usd, employment_status
  FROM snapshots
  WHERE kind = 'daily_econ'
  ORDER BY agent_id, tick_day
""")

if cash.empty or cash["tick_day"].nunique() < 2:
    print("Not enough daily snapshots to plot (need a multi-day run).")
    cash
else:
    fig, ax = plt.subplots(figsize=(11, 5))
    for agent_id, group in cash.groupby("agent_id"):
        ax.plot(group["tick_day"], group["cash_usd"], marker=".", label=agent_id)
    ax.set_xlabel("Day")
    ax.set_ylabel("Cash on hand (USD)")
    ax.set_title("Cash trajectory per persona")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.autofmt_xdate()
    plt.show()


## 8. Funnel — view → cart_add → purchase

Conversion rates per persona. Useful proxy for how deliberative each persona is.


In [ ]:
q("""
  SELECT
    agent_id,
    COUNT(*) FILTER (WHERE event_type = 'view')     AS views,
    COUNT(*) FILTER (WHERE event_type = 'cart_add') AS cart_adds,
    COUNT(*) FILTER (WHERE event_type = 'purchase') AS purchases,
    ROUND(100.0 * COUNT(*) FILTER (WHERE event_type = 'cart_add') /
      NULLIF(COUNT(*) FILTER (WHERE event_type = 'view'), 0), 1) AS view_to_cart_pct,
    ROUND(100.0 * COUNT(*) FILTER (WHERE event_type = 'purchase') /
      NULLIF(COUNT(*) FILTER (WHERE event_type = 'cart_add'), 0), 1) AS cart_to_purchase_pct
  FROM events
  GROUP BY 1
  ORDER BY 1
""")


## 9. Reflections — the LLM's self-talk

Weekly observations the `ReflectionEngine` produced about each persona. The raw text is rich — these are the LLM describing the persona's recent behavior in its own voice.


In [ ]:
import json

refls = con.execute("""
  SELECT agent_id, tick_day, payload
  FROM events WHERE event_type = 'reflection'
  ORDER BY tick_day, agent_id
""").fetchall()

if not refls:
    print("No reflections (need a run of at least 7 days).")
else:
    for agent_id, tick_day, payload in refls:
        observations = json.loads(payload).get("observations", [])
        print(f"--- {agent_id} on {tick_day} ---")
        for obs in observations:
            print(f"  - {obs}")
        print()


### Reflections — only one persona

Edit `AGENT` to follow one persona's trajectory week-over-week.


In [ ]:
AGENT = "persona-maya-001"

rows = con.execute(f"""
  SELECT tick_day, payload
  FROM events
  WHERE event_type = 'reflection' AND agent_id = '{AGENT}'
  ORDER BY tick_day
""").fetchall()
for tick_day, payload in rows:
    print(f"Week ending {tick_day}:")
    for obs in json.loads(payload).get('observations', []):
        print(f"  - {obs}")
    print()


## 10. Life events

Scripted life events that fired during this run (e.g., Maya's layoff in `electronics_shock.yaml`). LLM-driven life events are v1.


In [ ]:
life = con.execute("""
  SELECT agent_id, tick_day, payload
  FROM events WHERE event_type = 'life_event'
  ORDER BY tick_day, agent_id
""").fetchall()

if not life:
    print("No life events fired in this run.")
else:
    for agent_id, tick_day, payload in life:
        data = json.loads(payload)
        print(f"--- {agent_id} on {tick_day}: {data.get('event_id')} (LCU={data.get('lcu')}) ---")
        print(f"  Narration: {data.get('narration')}")
        print(f"  DNA diff:  {data.get('dna_diff')}")
        print()


## 11. Raw event browser

For drilling in. Edit the WHERE clause to filter by `agent_id`, `event_type`, `tick_day`, etc.


In [ ]:
q("""
  SELECT agent_id, tick_day, seq, event_type, payload
  FROM events
  WHERE agent_id = 'persona-maya-001'
  ORDER BY tick_day, seq
  LIMIT 50
""")
